# Geospatial Workflow Planning Agent Demo

This notebook demonstrates the `geospatial_workflow_planning_agent`, a GAS service that discovers published GAS capabilities and returns a client-side workflow plan.

The planning agent is intentionally **plan-only**. It can generate executable GAS Client Python code and a notebook skeleton, but it does not run those artifacts or call downstream GAS services itself. The client, notebook user, workflow platform, or AI orchestrator decides whether to execute the returned plan.

In this example, the planner is given a GAS Server `GetCapabilities` URL, asks that server for each agent's `DescribeAgent` document, matches available agents to a multi-step geospatial workflow, and returns the default planning artifacts:

- `interactive_workflow_graph`: HTML graph showing steps and dependencies
- `workflow_json`: machine-readable workflow plan
- `notebook_skeleton`: Jupyter notebook workflow skeleton

Additional options such as `human_readable` and `gas_client_python` can be requested explicitly with `plan_outputs`.


## Install and Import the GAS Client SDK

The public GAS Client SDK is available from PyPI as `gas-client`. If you are running this notebook inside the repository's local environment and already have the package installed, this cell is still safe to run.


In [ ]:
%pip install -q gas-client


In [ ]:
import json
import os
from pathlib import Path

import requests
from IPython.display import HTML, IFrame, Markdown, display

from gas_client import GasClient


## Configure the GAS Server and Credentials

Use your local GAS Server, a public deployment, or an ngrok URL. The planning agent needs an LLM credential because it interprets the user goal and matches it to discovered service capabilities.

The built-in GAS agents accept either `OPENAI_API_KEY` or `GIBD_API_KEY`. Other GAS servers may document different credential requirements in their own `DescribeAgent` documents.


In [ ]:
server_url = "http://127.0.0.1:4042"  # Replace with your public GAS Server URL if needed.

openai_api_key = os.getenv("OPENAI_API_KEY") or "YOUR_OPENAI_API_KEY"
gibd_api_key = os.getenv("GIBD_API_KEY") or None

client = GasClient(
    server_url,
    openai_api_key=openai_api_key,
    gibd_api_key=gibd_api_key,
    timeout=900,
)

planner_agent = client.agent("geospatial_workflow_planning_agent")


## Inspect the Planning Agent Description

`DescribeAgent` explains the planner's supported parameters, output options, and plan-only behavior.


In [ ]:
planner_description = planner_agent.describe()
print(planner_description["profile"]["name"])
print(planner_description["profile"]["description"])
print("\nSupported plan_outputs:")
print(json.dumps(planner_description["execute_task"]["parameters"]["plan_outputs"], indent=2))


## Build the Planning Request

The key parameter is `gas_servers`. Each item should be a GAS `GetCapabilities` URL. If this parameter is omitted, the planner uses the current server's local capability documents.

Here we explicitly provide the current server's `GetCapabilities` URL to demonstrate distributed capability discovery. The same pattern works with a remote GAS Server URL.


In [ ]:
capabilities_url = f"{server_url}/?SERVICE=GAS&VERSION=1.0.0&REQUEST=GetCapabilities"

workflow_goal = (
    "Download the DEM for Richland County, South Carolina, clip it to the Richland County boundary, "
    "randomly generate 100 points within the county, extract elevation values for those points, "
    "create a histogram of the sampled elevations, and display the generated points on top of the clipped DEM."
)

planning_parameters = {
    "gas_servers": [capabilities_url],
    "plan_outputs": [
        "interactive_workflow_graph",
        "workflow_json",
        "notebook_skeleton",
    ],
    "plan_detail": "executable",
    "include_validation_steps": True,
    "max_steps": 12,
}

print(capabilities_url)


## Execute the Planning Task in Streaming Mode

Streaming mode is useful because the planner reports discovery, matching, artifact generation, and completion progress while it works.

This cell builds the canonical GAS `ExecuteTask` request body first, prints the requested `plan_outputs`, and then sends that exact request with `execute_task_request(...)`. This makes it clear which output options are being sent to the server.


In [ ]:
planning_request = client.build_execute_task_request(
    workflow_goal,
    mode="stream",
    parameters=planning_parameters,
)

print("Requested plan outputs:")
print(json.dumps(planning_request["parameters"]["plan_outputs"], indent=2))

planning_result = None

for event in planner_agent.execute_task_request(
    planning_request,
    timeout=900,
):
    client.print_stream_event(event)
    if event.get("event") == "task_result":
        planning_result = event.get("payload")

if planning_result is None:
    raise RuntimeError("The stream ended before returning a task_result event.")

client.print_task_summary(planning_result)


## Use the Generated Plan

The planning agent has not executed the workflow. The next step is for a client, notebook user, workflow platform, or AI orchestrator to review the returned plan and optionally run the generated GAS Client Python or notebook skeleton.

Typical review checklist:

- Confirm each step selected the expected GAS agent.
- Confirm credentials required by each selected agent are available.
- Confirm dependencies and artifact passing are reasonable.
- Confirm validation checks are included for CRS, raster/vector overlap, join keys, and expected artifact formats.
- Run the generated code only after reviewing server URLs and credential placeholders.
